In [1]:
pip install --upgrade langchain langchain-google-genai langsmith google-generativeai pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip show langchain-google-genai

Name: langchain-google-genai
Version: 4.1.3
Summary: An integration package connecting Google's genai package and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/google
Author: 
Author-email: 
License: MIT
Location: /Users/nitishrmaladakar/anaconda3/envs/new_plant_env/lib/python3.11/site-packages
Requires: filetype, google-genai, langchain-core, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from typing import Dict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from langsmith import traceable
import os

In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [8]:


print(llm.invoke("hello from infosys email agent").content)

Hello there, Infosys email agent!

How can I assist you today? Are you here to optimize my inbox, provide some crucial information, or do you have a different query for me?


In [9]:
import sys
sys.path.append("../")

from src.tools import read_calendar, get_customer_profile
tools = {
"read_calendar": read_calendar,
"get_customer_profile": get_customer_profile
}

In [10]:
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")
emails.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [11]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
Classify this email into one of:
ignore
notify_human
respond
Email:
{email}
Return only the label.
"""
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}

In [12]:
def react_agent(state):
    email = state["email"]
    prompt = f"""
    You are an email assistant.
    You can use tools if needed.
    Tools:
    read_calendar
    get_customer_profile
    Email:
    {email}
    If you need a tool, write TOOL:<toolname>

    Otherwise give reply.
    """
    response = llm.invoke(prompt).content
    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}
    return {**state, "response": response}

In [13]:
from langgraph.graph import StateGraph
graph = StateGraph(dict)
graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)
def route(state):
    if state["triage"] == "respond":
        return "react"
    else:
        return "end"
graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")
app = graph.compile()

In [14]:
results = []
for _, row in emails.head(5).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})

    results.append({
    "email": email,
    "triage": output["triage"],
    "response": output.get("response", "")
    })

Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


In [15]:
results

[{'email': 'Reminder: The client meeting is scheduled at 10:00 tomorrow. Prepare your slides.',
  'triage': 'notify_human',
  'response': ''},
 {'email': 'Your invoice of INR 25515.09 is due on 2025-12-12. Please pay to avoid late fees.',
  'triage': 'notify_human',
  'response': ''},
 {'email': 'Reminder: The client meeting is scheduled at 11:30 tomorrow. Prepare your slides.',
  'triage': 'notify_human',
  'response': ''},
 {'email': 'Hello team, please find the attached weekly report and action items for the project.',
  'triage': 'notify_human',
  'response': ''},
 {'email': 'Hello team, please find the attached weekly report and action items for the project.',
  'triage': 'notify_human',
  'response': ''}]

In [16]:
pd.DataFrame(results).to_csv("../data/milestone1_new_output.csv", index=False)

In [19]:
merged = gold.merge(
pred,
on="email",
how="inner",
suffixes=("_gold", "_pred")
)

accuracy = (merged["expected"] ==
merged["triage"]).mean()
accuracy

nan

In [20]:
from langsmith import Client
client = Client()
client.list_projects()

<generator object Client.list_projects at 0x168a834c0>

In [21]:
list(client.list_projects())

[TracerSessionResult(id=UUID('7443d7cd-84a5-4d90-9dda-d0868b89fb8a'), start_time=datetime.datetime(2026, 1, 13, 22, 29, 20, 973000, tzinfo=datetime.timezone.utc), end_time=None, description=None, name='default', extra=None, tenant_id=UUID('cab35766-d1cf-4fef-892b-1df97382ff62'), reference_dataset_id=None, run_count=None, latency_p50=None, latency_p99=None, total_tokens=None, prompt_tokens=None, completion_tokens=None, last_run_start_time=None, feedback_stats=None, session_feedback_stats=None, run_facets=None, total_cost=None, prompt_cost=None, completion_cost=None, first_token_p50=None, first_token_p99=None, error_rate=None),
 TracerSessionResult(id=UUID('29127859-cfbf-4b30-9eea-817ebf165338'), start_time=datetime.datetime(2026, 1, 13, 13, 55, 28, 276591, tzinfo=datetime.timezone.utc), end_time=None, description=None, name='Infosys-Milestone-2', extra=None, tenant_id=UUID('cab35766-d1cf-4fef-892b-1df97382ff62'), reference_dataset_id=None, run_count=None, latency_p50=None, latency_p99=N